In [1]:
print("Hello World")

Hello World


In [1]:
from pathlib import Path
from utils.model_loader import ModelLoader
from src.rag_system.document_service import DocumentService   # adjust import
from src.rag_system.chroma_service_arc import ChromaManager       # adjust import
from config.config import settings
from src.rag_system.schemas import FileType

In [2]:
# Example: PDF file path
file_path = "C:\\Users\\302sy\\Desktop\\Generative AI\\StockSnapAI\\data\\attention_is_all_you_need.pdf"

# Process file with DocumentService
doc_service= DocumentService()
doc, meta = doc_service.process_single_file(file_path, FileType.PDF)
len(doc)


{"timestamp": "2025-09-04T07:52:38.175763Z", "level": "info", "event": "Processed PDF: 52 chunks from 15 pages"}


52

In [3]:
meta

{'pages': 15, 'has_images': False}

In [4]:
type(doc)

list

In [5]:
len(doc)

52

In [6]:
# Create ChromaManager
cm = ChromaManager(index_dir=Path("chroma_index/test_session"))

# Create / Load index
cm.load_or_create(doc)

# Add docs (deduplication handled)
added = cm.add_documents(doc)
print(f"Added {added} new docs to Chroma")


{"index_dir": "chroma_index\\test_session", "timestamp": "2025-09-04T07:52:42.922630Z", "level": "info", "event": "Loading existing Chroma index"}
Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Added 0 new docs to Chroma


In [7]:
## load to chroma manager

# Load retriever
retriever = cm.as_retriever(k=3)

# Load LLM
llm = ModelLoader().load_llm()

# Ask a question
query = "What is the main topic of this document?"
context_docs = retriever.invoke(query)

print("\nRetrieved Context:")
for i, d in enumerate(context_docs, 1):
    print(f"[Doc {i}]", d.page_content[:300])

# Send to LLM with context
context_text = "\n\n".join(d.page_content for d in context_docs)
prompt = f"Based on the following document context, answer the question:\n\n{context_text}\n\nQuestion: {query}"

response = llm.invoke(prompt)
print("\nLLM Answer:", response.content)

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



Retrieved Context:
[Doc 1] Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



LLM Answer: The main topic of this document is the introduction of a new network architecture called the Transformer, which is based solely on attention mechanisms for sequence transduction tasks. The paper highlights the advantages of the Transformer over traditional models that use recurrent or convolutional neural networks, showing that it achieves superior performance in machine translation tasks while being more efficient in terms of training time and resource requirements.


In [ ]:
chunks